# TT-18 — 02. Gradient Boosting Regressor: baseline → AVM co khoang gia
Huan luyen baseline, Gradient Boosting voi early stopping, hoi quy phan vi (khoang gia), median APE, va co che human-in-the-loop.

In [1]:
import sys
sys.path.append('../src')
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
import features as feat
from train import median_ape, rmse_log

RANDOM_STATE = 42
df_raw = feat.load_data('../data/train.csv')
df_clean = feat.handle_missing(df_raw)
df_fe = feat.engineer_features(df_clean)
y = df_fe['SalePrice'].astype(float)
y_log = np.log1p(y)

X_train_df, X_val_df, y_train, y_val, y_train_log, y_val_log = train_test_split(
    df_fe.drop(columns=['SalePrice']), y, y_log, test_size=0.2, random_state=RANDOM_STATE)

preprocessor, cols = feat.build_preprocessor(df_fe)
X_train = preprocessor.fit_transform(X_train_df)
X_val = preprocessor.transform(X_val_df)
X_train.shape, X_val.shape

((1168, 56), (292, 56))

## 1. Baseline: Dummy / Linear / Ridge (tren thang log)

In [2]:
baseline_results = {}
for name, model in [
    ('DummyRegressor', DummyRegressor(strategy='mean')),
    ('LinearRegression', LinearRegression()),
    ('Ridge', Ridge(alpha=10.0)),
]:
    model.fit(X_train, y_train_log)
    score = rmse_log(y_val_log, model.predict(X_val))
    baseline_results[name] = score
    print(f'{name:20s} RMSE(log) = {score:.4f}')

DummyRegressor       RMSE(log) = 0.3446
LinearRegression     RMSE(log) = 0.1067
Ridge                RMSE(log) = 0.1072


## 2. Gradient Boosting Regressor + Early Stopping

In [3]:
gbr = GradientBoostingRegressor(
    n_estimators=1000, learning_rate=0.03, max_depth=3,
    subsample=0.8, max_features='sqrt',
    validation_fraction=0.1, n_iter_no_change=50, tol=1e-4,
    random_state=RANDOM_STATE,
)
gbr.fit(X_train, y_train_log)
n_trees_used = gbr.n_estimators_
gbr_rmse_log = rmse_log(y_val_log, gbr.predict(X_val))
print(f'RMSE(log) = {gbr_rmse_log:.4f}  |  dung sau {n_trees_used} cay (early stopping)')

RMSE(log) = 0.1244  |  dung sau 661 cay (early stopping)


## 3. Duong train/validation loss theo so cay → diem overfit

In [4]:
gbr_curve = GradientBoostingRegressor(
    n_estimators=1000, learning_rate=0.03, max_depth=3,
    subsample=0.8, max_features='sqrt', random_state=RANDOM_STATE,
)
gbr_curve.fit(X_train, y_train_log)
train_loss = gbr_curve.train_score_
val_loss = np.array([mean_squared_error(y_val_log, p) for p in gbr_curve.staged_predict(X_val)])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_loss, label='Train loss (deviance)')
ax.plot(val_loss, label='Validation loss (MSE, thang log)')
ax.axvline(n_trees_used, color='gray', linestyle='--', label=f'Early stop @ {n_trees_used} cay')
ax.set_xlabel('So cay (boosting stage)'); ax.set_ylabel('Loss')
ax.set_title('Train vs Validation loss theo so cay')
ax.legend(); fig.tight_layout()
fig.savefig('../reports/loss_theo_so_cay.png', dpi=130)
plt.show()

## 4. Feature engineering: truoc / sau `TotalSF`, `TuoiNha`, `DaSuaChua`

In [5]:
fe_cols = ['TotalSF', 'TuoiNha', 'DaSuaChua']
df_no_fe = df_clean.drop(columns=[c for c in fe_cols if c in df_clean.columns], errors='ignore')
pre_no_fe, _ = feat.build_preprocessor(df_no_fe)
Xtr_no_fe = pre_no_fe.fit_transform(X_train_df.drop(columns=fe_cols, errors='ignore'))
Xval_no_fe = pre_no_fe.transform(X_val_df.drop(columns=fe_cols, errors='ignore'))

def quick_gbr():
    return GradientBoostingRegressor(n_estimators=400, learning_rate=0.05, max_depth=3,
                                       subsample=0.8, max_features='sqrt', random_state=RANDOM_STATE)

m_no_fe = quick_gbr().fit(Xtr_no_fe, y_train_log)
rmse_no_fe = rmse_log(y_val_log, m_no_fe.predict(Xval_no_fe))
m_fe = quick_gbr().fit(X_train, y_train_log)
rmse_fe = rmse_log(y_val_log, m_fe.predict(X_val))

improve = 100 * (rmse_no_fe - rmse_fe) / rmse_no_fe
print(f'RMSE(log) truoc khi them dac trung: {rmse_no_fe:.4f}')
print(f'RMSE(log) sau khi them dac trung:   {rmse_fe:.4f}')
print(f'Cai thien: {improve:.1f}%')

RMSE(log) truoc khi them dac trung: 0.1253
RMSE(log) sau khi them dac trung:   0.1243
Cai thien: 0.8%


## 5. Hoi quy phan vi (10 / 50 / 90) → khoang gia

In [6]:
quantile_models = {}
for q in [0.1, 0.5, 0.9]:
    qm = GradientBoostingRegressor(loss='quantile', alpha=q, n_estimators=500,
                                     learning_rate=0.05, max_depth=3, subsample=0.8,
                                     random_state=RANDOM_STATE)
    qm.fit(X_train, y_train_log)
    quantile_models[q] = qm

pred_p10 = np.expm1(quantile_models[0.1].predict(X_val))
pred_p50 = np.expm1(quantile_models[0.5].predict(X_val))
pred_p90 = np.expm1(quantile_models[0.9].predict(X_val))
pred_p10, pred_p90 = np.minimum(pred_p10, pred_p90), np.maximum(pred_p10, pred_p90)

coverage = float(np.mean((y_val.values >= pred_p10) & (y_val.values <= pred_p90)) * 100)
print(f'Coverage thuc te cua khoang 10-90%: {coverage:.1f}% (muc tieu ~80%)')

Coverage thuc te cua khoang 10-90%: 63.7% (muc tieu ~80%)


In [7]:
order = np.argsort(y_val.values)[:80]
x_axis = np.arange(len(order))
fig, ax = plt.subplots(figsize=(9, 6))
ax.fill_between(x_axis, pred_p10[order], pred_p90[order], color='#5bc0de', alpha=0.3, label='Khoang 10-90%')
ax.plot(x_axis, pred_p50[order], color='#0275d8', label='Du doan p50')
ax.scatter(x_axis, y_val.values[order], color='#d9534f', s=12, label='Gia that', zorder=5)
ax.set_xlabel('Can nha (sap xep theo gia that)'); ax.set_ylabel('SalePrice')
ax.set_title(f'Khoang gia du bao 10-90% (coverage: {coverage:.1f}%)')
ax.legend(); fig.tight_layout()
fig.savefig('../reports/khoang_gia.png', dpi=130)
plt.show()

## 6. Median APE — chuan nganh AVM (< 10-12%)

In [8]:
m_ape = median_ape(y_val.values, pred_p50)
ape_all = np.abs((y_val.values - pred_p50) / y_val.values) * 100
print(f'Median APE = {m_ape:.2f}%')

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(ape_all, bins=30, color='#5cb85c', edgecolor='white')
ax.axvline(m_ape, color='#d9534f', linestyle='--', label=f'Median APE = {m_ape:.1f}%')
ax.set_xlabel('Absolute Percentage Error (%)'); ax.set_ylabel('So can nha')
ax.set_title('Phan phoi APE tren tap validation')
ax.legend(); fig.tight_layout()
fig.savefig('../reports/ape_distribution.png', dpi=130)
plt.show()

Median APE = 8.39%


## 7. Human-in-the-loop
Neu khoang du bao (p90 − p10) rong hon **25%** gia diem (p50), ho so duoc tu dong chuyen cho tham dinh vien thay vi duyet tu dong.

In [9]:
interval_width_pct = (pred_p90 - pred_p10) / pred_p50 * 100
needs_human = interval_width_pct > 25
pct_auto = float(np.mean(~needs_human) * 100)

hitl_table = pd.DataFrame([
    {'nhom': 'Tu dong duyet (khoang <=25%)', 'ty_le_%': round(pct_auto, 1),
     'so_ho_so': int((~needs_human).sum()),
     'median_ape_%': round(median_ape(y_val.values[~needs_human], pred_p50[~needs_human]), 2) if (~needs_human).any() else np.nan},
    {'nhom': 'Chuyen tham dinh vien (khoang >25%)', 'ty_le_%': round(100 - pct_auto, 1),
     'so_ho_so': int(needs_human.sum()),
     'median_ape_%': round(median_ape(y_val.values[needs_human], pred_p50[needs_human]), 2) if needs_human.any() else np.nan},
])
hitl_table.to_csv('../reports/human_in_the_loop.csv', index=False)
hitl_table

,nhom,ty_le_%,so_ho_so,median_ape_%
0,Tu dong duyet (khoang <=25%),42.5,124,8.41
1,Chuyen tham dinh vien (khoang >25%),57.5,168,8.39


## 8. So sanh thoi gian train: GradientBoosting vs HistGradientBoosting

In [10]:
t0 = time.time()
gbr_timing = GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=RANDOM_STATE)
gbr_timing.fit(X_train, y_train_log)
t_gbr = time.time() - t0

t0 = time.time()
hgbr = HistGradientBoostingRegressor(max_iter=300, max_depth=3, learning_rate=0.05, random_state=RANDOM_STATE)
hgbr.fit(X_train, y_train_log)
t_hgbr = time.time() - t0

print(f'GradientBoostingRegressor:     {t_gbr:.2f}s | RMSE(log)={rmse_log(y_val_log, gbr_timing.predict(X_val)):.4f}')
print(f'HistGradientBoostingRegressor: {t_hgbr:.2f}s | RMSE(log)={rmse_log(y_val_log, hgbr.predict(X_val)):.4f}')

GradientBoostingRegressor:     1.15s | RMSE(log)=0.1247
HistGradientBoostingRegressor: 0.18s | RMSE(log)=0.1224


## 9. Luu pipeline cuoi cung

In [11]:
full_pipeline = Pipeline([('preprocess', preprocessor), ('model', gbr)])
joblib.dump(full_pipeline, '../models/gbr_pipeline.joblib')
joblib.dump(quantile_models, '../models/gbr_quantile_models.joblib')
print('Da luu ../models/gbr_pipeline.joblib va ../models/gbr_quantile_models.joblib')

Da luu ../models/gbr_pipeline.joblib va ../models/gbr_quantile_models.joblib


## Tom tat
- **Median APE**, **coverage 10–90%**, va **ty le tu dong hoa** duoc in o cac buoc tren.
- Xem `README.md` cua project de doi chieu voi tieu chi hoan thanh.